In [1]:
import json
from py2neo import Graph, Node, Relationship

In [2]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "root1234"

In [3]:
JSON_PATH = "Cyber_security_experts_Manual_scraping_ENRICHED.json"


In [4]:
graph = Graph(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

In [5]:
def sanitize_skill(raw_skill):
    """
    Remove leading '-' or spaces, trim whitespace,
    and return None if it's 'Null' or empty after cleaning.
    """
    if not raw_skill or raw_skill.strip().lower() == "null":
        return None
    cleaned = raw_skill.lstrip("- ").strip()
    if not cleaned:
        return None
    return cleaned

In [6]:
person_cache = {}
experience_cache = {}
org_cache = {}
skill_cache = {}

In [9]:
def load_experts_into_neo4j():
    # Load experts from JSON file into Neo4j.
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        experts_data = json.load(f)

    person_cache = {}
    experience_cache = {}
    org_cache = {}
    skill_cache = {}

    tx = graph.begin()  # Start transaction
    batch_size = 100
    count = 0  # Track number of inserts before committing

    for person_data in experts_data:
        person_name = person_data.get("name", "").strip()
        person_id = person_name.lower().replace(" ", "_")

        # Create Person node (if not already in cache)
        if person_id not in person_cache:
            person_node = Node("Person", person_id=person_id, name=person_name)
            tx.merge(person_node, "Person", "person_id")
            person_cache[person_id] = person_node
            print(f"[Person] Created: {person_name}")

        person_node = person_cache[person_id]

        # Add top-level skills (Person → Skill)
        for raw_skill in person_data.get("skills", []):
            skill_name = sanitize_skill(raw_skill)
            if skill_name:
                skill_id = skill_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

                if skill_id not in skill_cache:
                    skill_node = Node("Skill", skill_id=skill_id, name=skill_name)
                    tx.merge(skill_node, "Skill", "skill_id")
                    skill_cache[skill_id] = skill_node
                    print(f"[Skill] Created: {skill_name}")

                skill_node = skill_cache[skill_id]
                tx.merge(Relationship(person_node, "HAS_SKILL", skill_node))

        # Process Experience nodes
        for idx, exp in enumerate(person_data.get("experiences", [])):
            exp_role = exp.get("role", "").strip()
            exp_workplace = exp.get("workplace", "").strip()
            exp_duration = exp.get("duration", "").strip()
            exp_description = exp.get("Description", "").strip()
            experience_id = f"{person_id}_exp_{idx}"

            if experience_id not in experience_cache:
                experience_node = Node("Experience",
                                       experience_id=experience_id,
                                       role=exp_role,
                                       duration=exp_duration,
                                       description=exp_description)
                tx.merge(experience_node, "Experience", "experience_id")
                experience_cache[experience_id] = experience_node
                print(f"[Experience] Created: {exp_role} at {exp_workplace}")

            experience_node = experience_cache[experience_id]
            tx.merge(Relationship(person_node, "HAS_EXPERIENCE", experience_node))

            # Create Organization node
            if exp_workplace:
                org_id = exp_workplace.lower().replace(" ", "_")
                if org_id not in org_cache:
                    org_node = Node("Organization", organization_id=org_id, name=exp_workplace)
                    tx.merge(org_node, "Organization", "organization_id")
                    org_cache[org_id] = org_node
                    print(f"[Organization] Created: {exp_workplace}")

                org_node = org_cache[org_id]
                tx.merge(Relationship(experience_node, "AT_ORGANIZATION", org_node))

            # Add Experience-specific Skills
            for raw_skill in exp.get("skills_extracted", []):
                skill_name = sanitize_skill(raw_skill)
                if skill_name:
                    skill_id = skill_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

                    if skill_id not in skill_cache:
                        skill_node = Node("Skill", skill_id=skill_id, name=skill_name)
                        tx.merge(skill_node, "Skill", "skill_id")
                        skill_cache[skill_id] = skill_node
                        print(f"[Skill] Created: {skill_name}")

                    skill_node = skill_cache[skill_id]
                    tx.merge(Relationship(experience_node, "USED_SKILL", skill_node))

        # Commit every `batch_size` inserts to avoid transaction overload
        count += 1
        if count % batch_size == 0:
            tx.commit()
            tx = graph.begin()
            print(f"Committed {count} entries...")

    # Final commit
    tx.commit()
    print("All data loaded successfully!")

if __name__ == "__main__":
    load_experts_into_neo4j()

[Person] Created: Adam Evans
[Skill] Created: Web Development
[Skill] Created: Ethical Hacking
[Skill] Created: Penetration Testing
[Skill] Created: Malware Analysis
[Skill] Created: Reverse Engineering
[Skill] Created: Vulnerability Assessment
[Skill] Created: Network Security
[Skill] Created: Firewall Management
[Skill] Created: Intrusion Detection
[Skill] Created: System Administration
[Skill] Created: Network Administration
[Skill] Created: Security Operations
[Skill] Created: Storage Area Networks
[Skill] Created: Backtrack
[Skill] Created: EnCase
[Skill] Created: FTK
[Skill] Created: Metasploit
[Skill] Created: Incident Management
[Skill] Created: Incident Analysis
[Skill] Created: Memory Forensics
[Skill] Created: Assembly Language
[Skill] Created: Static Code Analysis
[Skill] Created: Computer Forensics
[Skill] Created: IPS
[Skill] Created: CEH
[Skill] Created: Firewalls
[Skill] Created: Information Security Management
[Skill] Created: PKI
[Skill] Created: ISO 27001
[Skill] Cre

[Organization] Created: IBM Security Response Team
[Skill] Created: Defined security methodologies
[Skill] Created: Developed security services
[Skill] Created: Security Response Team operations
[Skill] Created: Consulting across internal and external systems
[Experience] Created: IT Architect / Software Engineer at IBM Global E-Technology Center
[Organization] Created: IBM Global E-Technology Center
[Skill] Created: Compliance reviews
[Skill] Created: PKI/CA design
[Skill] Created: Digital certificate project
[Experience] Created: Systems Management Specialist at Integrated Systems Solutions Corp (IBM subsidiary)
[Organization] Created: Integrated Systems Solutions Corp (IBM subsidiary)
[Skill] Created: Security-compliance tools development
[Experience] Created: Programmer/Network Specialist, Third Party Development Coordinator, and Sales Engineer Technician at Florida businesses
[Organization] Created: Florida businesses
[Person] Created: Andy Matthiesen
[Skill] Created: Project Mana

[Skill] Created: FINRA
[Skill] Created: FFIEC
[Skill] Created: Digital Transformation
[Skill] Created: GLBA
[Skill] Created: Network Vulnerability Assessment
[Skill] Created: Webinar
[Experience] Created: CEO | Founder at Mijares Consulting
[Organization] Created: Mijares Consulting
[Experience] Created: VP Risk & Information Security at UDT
[Organization] Created: UDT
[Experience] Created: IT Risk Director for Banks at Kaufman Rossin
[Organization] Created: Kaufman Rossin
[Experience] Created: Risk Advisory Services Manager at Kaufman Rossin
[Experience] Created: Senior IT Consultant at Advisory Financial Group
[Organization] Created: Advisory Financial Group
[Experience] Created: Senior IT Auditor at PwC
[Organization] Created: PwC
[Experience] Created: IT Consultant at McGladrey
[Organization] Created: McGladrey
[Experience] Created: IT Consultant at Deloitte
[Organization] Created: Deloitte
[Experience] Created: Consultant at Deloitte
[Person] Created: Amit Basu
[Skill] Created: Ta

[Skill] Created: IoT security
[Skill] Created: US-EU Privacy Shield compliance
[Skill] Created: GDPR compliance
[Skill] Created: Cyber and privacy training
[Skill] Created: Information Security Council
[Skill] Created: Security stack optimization
[Experience] Created: Secretary, Board Of Directors at Cloud Security Alliance – Detroit
[Organization] Created: Cloud Security Alliance – Detroit
[Skill] Created: Identity as the Digital Perimeter
[Skill] Created: Best practices in InfoSec community
[Experience] Created: Global Information Security Officer at Nexteer Automotive
[Skill] Created: Enterprise security risk assessments
[Skill] Created: Designing security vision and roadmap
[Skill] Created: Key risk-mitigation initiatives
[Skill] Created: Strengthening security boundary
[Experience] Created: Founder & Principal Consultant at ProFortis Solutions, LLC
[Organization] Created: ProFortis Solutions, LLC
[Skill] Created: HIPAA-HITECH compliance
[Experience] Created: Chief Information Offi

[Skill] Created: CI/CD Security Automation
[Skill] Created: SBOM (Software Bill of Materials)
[Skill] Created: Third-party Open Source Security
[Skill] Created: Red Team Exercises
[Skill] Created: Purple Teaming
[Skill] Created: Zero-day Incident Response
[Skill] Created: Metrics and KPIs in Security
[Experience] Created: Director Application Security, Vulnerability Management, Security Engineering and Red Team at Fannie Mae
[Skill] Created: Architecting security tools and processes
[Skill] Created: DevSecOps integration
[Skill] Created: Security testing
[Skill] Created: On-prem and AWS integration
[Skill] Created: Continuous application scanning
[Skill] Created: Zero-day vulnerabilities analysis
[Skill] Created: Monitoring third-party libraries
[Skill] Created: Penetration tests
[Skill] Created: SME for audits
[Experience] Created: Senior Manager Application Security and Engineering at Fannie Mae
[Skill] Created: Secure SDLC standards
[Skill] Created: Security development methodologie

...

[Experience] Created: Director, First Vice President, Global Head of Information Security Operations at ABN AMRO Bank, N.V.
[Organization] Created: ABN AMRO Bank, N.V.
[Person] Created: Todd Fitzgerald
[Skill] Created: Programme Governance
[Skill] Created: Department Start-up
[Skill] Created: ITIL v3 Foundations Certified
[Skill] Created: Published Author
[Skill] Created: Fortune 500
[Skill] Created: Senior Management Communications
[Skill] Created: SOX 404
[Experience] Created: Adjunct Professor, Cybersecurity Leadership at McCormick School of Engineering
[Organization] Created: McCormick School of Engineering
[Experience] Created: CISO Leadership Author/Advisor, Keynote Speaker at CISO SPOTLIGHT, LLC
[Organization] Created: CISO SPOTLIGHT, LLC
[Experience] Created: Vice President, Cybersecurity Strategy at CyberRisk Collaborative
[Experience] Created: Host, The CISO Stories Podcast at The CISO Stories Podcast
[Organization] Created: The CISO Stories Podcast
[Experience] Created: Seni

[Organization] Created: SIMBUS, LLC
[Experience] Created: Impact Creator, Thought Leadership Contributor Program at IEEE
[Organization] Created: IEEE
[Experience] Created: Member, Privacy & Security Architecture for Consumer Wireless Devices Working Group at IEEE COMSOC Par 1912
[Organization] Created: IEEE COMSOC Par 1912
[Experience] Created: Distinguished Fellow at Ponemon Institute
[Organization] Created: Ponemon Institute
[Experience] Created: Advisor & Subject Matter Expert at 3M Privacy Consultant
[Organization] Created: 3M Privacy Consultant
[Experience] Created: Advisory Board Member at DFLabs – Cyber Incidents Under Control
[Organization] Created: DFLabs – Cyber Incidents Under Control
[Experience] Created: Advisory Board Member at Westchester Biotech Project
[Organization] Created: Westchester Biotech Project
[Experience] Created: Advisory Board Member at Anonos: BigPrivacy Unlocks Data Analytics
[Organization] Created: Anonos: BigPrivacy Unlocks Data Analytics
[Experience] 

[Experience] Created: Defense Innovation Scholar - Cybersecurity and Misuse of AI (Project Wǎng Yuè Lead) at Stanford Gordian Knot Center for National Security Innovation
[Organization] Created: Stanford Gordian Knot Center for National Security Innovation
[Experience] Created: Co-Lead of DiplomAItrics Initiative at Stanford Social Media Lab
[Organization] Created: Stanford Social Media Lab
[Experience] Created: AI Security Expert at Duco
[Organization] Created: Duco
[Experience] Created: GenAI and Child Safety - Model Development Co-Lead at IEEE
[Experience] Created: AI Model Training - Content Policy at Freeman Spogli Institute for International Studies
[Organization] Created: Freeman Spogli Institute for International Studies
[Experience] Created: Cyber Threat Intelligence and Data Science Intern at Resilience
[Experience] Created: Trusted Election Analysis, AI and Information Quality Co-Lead at Harvard Belfer Center
[Organization] Created: Harvard Belfer Center
[Experience] Created

[Skill] Created: International Project Management
[Skill] Created: Innovative Thinking
[Skill] Created: Cyber Awareness Training
[Skill] Created: Awareness
[Experience] Created: Field Chief Information Security Officer (CISO) for Public Sector at Presidio
[Experience] Created: Chief Strategist & Chief Security Officer at Security Mentor, Inc.
[Organization] Created: Security Mentor, Inc.
[Experience] Created: Chief Security Officer (CSO) at State of Michigan
[Organization] Created: State of Michigan
[Experience] Created: Chief Technology Officer & Deputy Director, Infrastructure Services at Michigan Department of Technology, Management & Budget
[Organization] Created: Michigan Department of Technology, Management & Budget
[Experience] Created: Senior Technology Executive – e-Michigan Office at State of Michigan
[Experience] Created: CIO – Department of Management & Budget (DMB) at State of Michigan
[Experience] Created: Technical Director at ManTech
[Experience] Created: Senior Network

/var/folders/zq/w3gtlndx0qsdt92gmtmqdnkm0000gn/T/ipykernel_92052/20587005.py:99: DeprecationWarning: The transaction.commit() method is deprecated, use graph.commit(transaction) instead
  tx.commit()


In [10]:
print("done")

done


In [12]:
from py2neo import Graph
graph = Graph("bolt://localhost:7687", auth=("neo4j", "root1234"))
print(graph.run("RETURN 1").data())

[{'1': 1}]


In [13]:
import json
from py2neo import Graph, Node, Relationship

# ---------------------
# CONFIGURATION
# ---------------------
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "root1234"
JSON_PATH = "CyberSecuirty_experts_ENRICHED.json"

# ---------------------
# CONNECT TO NEO4J
# ---------------------
graph = Graph(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# ---------------------
# LOAD JSON DATA
# ---------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total records loaded: {len(data)}")

# ---------------------
# PROCESS JSON OBJECTS
# ---------------------
person_cache = {}
experience_cache = {}
org_cache = {}
skill_cache = {}

for person_data in data:
    person_name = person_data.get("name", "").strip()
    if not person_name:
        continue  # Skip empty names

    person_id = person_name.lower().replace(" ", "_")

    # ---- CREATE PERSON NODE ----
    if person_id not in person_cache:
        person_node = Node("Person", person_id=person_id, name=person_name)
        graph.merge(person_node, "Person", "person_id")
        person_cache[person_id] = person_node
        print(f"[Person] Created: {person_name}")

    person_node = person_cache[person_id]

    # ---- ADD TOP-LEVEL PERSON SKILLS ----
    for raw_skill in person_data.get("skills", []):
        skill_name = raw_skill.strip()
        if skill_name and skill_name.lower() != "null":
            skill_id = skill_name.lower().replace(" ", "_")
            if skill_id not in skill_cache:
                skill_node = Node("Skill", skill_id=skill_id, name=skill_name)
                graph.merge(skill_node, "Skill", "skill_id")
                skill_cache[skill_id] = skill_node
                print(f"[Skill] Created: {skill_name}")

            graph.merge(Relationship(person_node, "HAS_SKILL", skill_cache[skill_id]))

    # ---- PROCESS EXPERIENCES ----
    for idx, exp in enumerate(person_data.get("experiences", [])):
        exp_role = exp.get("role", "").strip()
        exp_workplace = exp.get("workplace", "").strip()
        exp_duration = exp.get("duration", "").strip()
        exp_description = exp.get("Description", "").strip()
        experience_id = f"{person_id}_exp_{idx}"

        if experience_id not in experience_cache:
            experience_node = Node("Experience",
                                   experience_id=experience_id,
                                   role=exp_role,
                                   duration=exp_duration,
                                   description=exp_description)
            graph.merge(experience_node, "Experience", "experience_id")
            experience_cache[experience_id] = experience_node
            print(f"[Experience] {exp_role} at {exp_workplace} added.")

        experience_node = experience_cache[experience_id]
        graph.merge(Relationship(person_node, "HAS_EXPERIENCE", experience_node))

        # ---- ORGANIZATION ----
        if exp_workplace:
            org_id = exp_workplace.lower().replace(" ", "_")
            if org_id not in org_cache:
                org_node = Node("Organization", organization_id=org_id, name=exp_workplace)
                graph.merge(org_node, "Organization", "organization_id")
                org_cache[org_id] = org_node
                print(f"[Organization] {exp_workplace} added.")

            graph.merge(Relationship(experience_node, "AT_ORGANIZATION", org_cache[org_id]))

        # ---- EXPERIENCE-LEVEL SKILLS ----
        for raw_skill in exp.get("skills_extracted", []):
            skill_name = raw_skill.strip()
            if skill_name and skill_name.lower() != "null":
                skill_id = skill_name.lower().replace(" ", "_")
                if skill_id not in skill_cache:
                    skill_node = Node("Skill", skill_id=skill_id, name=skill_name)
                    graph.merge(skill_node, "Skill", "skill_id")
                    skill_cache[skill_id] = skill_node
                    print(f"[Skill] Created: {skill_name}")

                graph.merge(Relationship(experience_node, "USED_SKILL", skill_cache[skill_id]))

print("✅ Expert KG successfully loaded into Neo4j!")


Total records loaded: 203
[Person] Created: Kristine Amyan
[Skill] Created: Customer Relationship Management (CRM)
[Skill] Created: Shopify
[Skill] Created: E-Commerce Strategy
[Experience] Marketing Manager at CyberDuo added.
[Organization] CyberDuo added.
[Experience] Copywriter/Marketing Coordinator at CyberDuo added.
[Skill] Created: - SMM (Social Media Marketing) in cybersecurity
[Skill] Created: - Content marketing in cybersecurity
[Experience] Office Assistant at Berenji & Associates added.
[Organization] Berenji & Associates added.
[Experience] Sales Associate at TJ Maxx added.
[Organization] TJ Maxx added.
[Person] Created: Anna (wanqing) Li
[Skill] Created: SQL
[Experience]  at Deputy Technical Director added.
[Organization] Deputy Technical Director added.
[Skill] Created: - Monitoring and Response
[Skill] Created: - Governance
[Skill] Created: - Strategic Planning
[Skill] Created: - Risk Mitigation
[Skill] Created: - Technical Vision Development
[Skill] Created: - Risk Iden

[Skill] Created: Trend Forecasting
[Skill] Created: Supervisory Skills
[Skill] Created: Construction
[Experience] Director of Technical Department at ExtraHop added.
[Organization] ExtraHop added.
[Experience] Department Manager at ExtraHop added.
[Experience] System Engineer at ExtraHop added.
[Skill] Created: - Network infrastructure
[Skill] Created: - Cloud environments
[Skill] Created: - Security tools
[Person] Created: Mia Thomas
[Skill] Created: Analysis  SQL
[Experience] Director of Technical Department at ExtraHop added.
[Skill] Created: - Network Security
[Skill] Created: - Risk Management
[Experience] Department Manager at ExtraHop added.
[Experience] System Engineer at ExtraHop added.
[Person] Created: Mia Rodriguez
[Experience] Director of Technical Department at ExtraHop added.
[Experience] Department Manager at ExtraHop added.
[Skill] Created: - Analyze customer network needs
[Skill] Created: - Promote application of ExtraHop
[Skill] Created: - Provide targeted assistance

[Skill] Created: - Development of business/IT continuity plans
[Skill] Created: - Maintenance of technical controls for security
[Skill] Created: - Identification and implementation of mitigations for system risk
[Skill] Created: - Participate in IT change management process
[Experience] Director of IT at Harbour Learning Trust Â· Full-time added.
[Organization] Harbour Learning Trust Â· Full-time added.
[Skill] Created: - Information management
[Skill] Created: - IT strategy development
[Skill] Created: - IT strategy implementation
[Skill] Created: - IT compliance
[Skill] Created: - IT systems monitoring
[Skill] Created: - Data analysis
[Skill] Created: - Data interpretation
[Skill] Created: - IT advisory
[Skill] Created: - IT environment integrity
[Skill] Created: - IT business continuity
[Skill] Created: - IT procurement
[Skill] Created: - IT systems management
[Skill] Created: - IT incident management
[Experience] Group IT Infrastructrue Manager at Inspire Education Group added.
[O

[Skill] Created: - Incident response
[Skill] Created: - Crisis management
[Skill] Created: - Security advisory
[Experience] Incident Response Coordinator at Northwave. Intelligent Security Operations added.
[Organization] Northwave. Intelligent Security Operations added.
[Skill] Created: - Incident response management
[Skill] Created: - Communication with stakeholders
[Skill] Created: - Security incident preparation
[Experience] Account Manager a.i. at Northwave. Intelligent Security Operations added.
[Experience] Incident Responder at Northwave. Intelligent Security Operations added.
[Skill] Created: - Incident management
[Skill] Created: - Security operations center (SOC) operations
[Experience] Chairman at Alumnivereniging ENIAC added.
[Organization] Alumnivereniging ENIAC added.
[Experience] Cyber Security Consultant at Ordina added.
[Organization] Ordina added.
[Skill] Created: - Security expertise
[Skill] Created: - Security and privacy awareness
[Experience] Security Consultant 

[Skill] Created: - Cybersecurity strategy development
[Skill] Created: - Cyber risk management
[Skill] Created: - Data protection and privacy
[Skill] Created: - Payments security
[Skill] Created: - Third-party cybersecurity management
[Skill] Created: - Data security risk management
[Experience] US Cyber Advise & Implement Leader at Deloitte added.
[Experience] US Cyber Secure Portfolio Leader at Deloitte added.
[Experience] US Cyber Resilient Offering Leader at Deloitte added.
[Experience] Product Manager at VeriSign added.
[Organization] VeriSign added.
[Experience] Consulting Manager at Exault added.
[Organization] Exault added.
[Experience] Consultant at Accenture added.
[Organization] Accenture added.
[Person] Created: Dr. Jason Edwards, DMIST, CISSP, CRISC
[Experience] Principal, Security Growth Strategy at Amazon · Full-time added.
[Organization] Amazon · Full-time added.
[Experience] Adjunct Professor/Course Developer - IT/Cybersecurity at Hallmark University · Contract added.


...

[Skill] Created: - Solution definition and incubation
[Skill] Created: - Analyst community engagement
[Skill] Created: - Publication management (State of Cybersecurity Report)
[Experience] Chief Editor - Wipro's State of Cybersecurity Report at Wipro Limited added.
[Skill] Created: - Macro, Micro, Meso and Future Views of Cybersecurity
[Skill] Created: - Trends in nation state attacks
[Skill] Created: - Malware
[Skill] Created: - Vulnerabilities
[Skill] Created: - Cyber regulations
[Skill] Created: - Patent research
[Experience] Practice Lead- Enterprise Mobile Security Services at Wipro Technologies added.
[Skill] Created: - Mobile Security
[Skill] Created: - Security Testing (mobile apps)
[Skill] Created: - Mobile Device Management (MDM)
[Experience] Practice Lead - Security Strategy & Architecture Services at Wipro Technologies added.
[Skill] Created: - Security risk management
[Skill] Created: - IT Security Strategy & Architecture
[Skill] Created: - Cross-domain security knowledge 

[Experience] Information Security Engineer at Pacific Sunwear (US) added.
[Organization] Pacific Sunwear (US) added.
[Experience] System & Process Assurance Manager at PwC (Vietnam) added.
[Organization] PwC (Vietnam) added.
[Experience] IS Assurance Associate at BDO (US) added.
[Organization] BDO (US) added.
[Person] Created: Danny Turner
[Skill] Created: Governance
[Experience] Manager at KPMG US · Full-time added.
[Experience] Associate Director at KPMG Australia added.
[Organization] KPMG Australia added.
[Experience] Manager at KPMG Australia added.
[Experience] Senior Consultant at KPMG Australia added.
[Experience] Technical Support at Snowy Hydro Limited added.
[Organization] Snowy Hydro Limited added.
[Person] Created: Abhi Shahi
[Skill] Created: Requirements Analysis
[Experience] Senior Manager at ING Australia · Full-time added.
[Organization] ING Australia · Full-time added.
[Skill] Created: Technology risk consultation
[Experience] Associate Director - Technology Risk and 

[Skill] Created: - IDS/IPS infrastructure
[Skill] Created: - Cisco Secure Access Control System
[Skill] Created: - Implementation of IDS/IPS
[Skill] Created: - Symantec Endpoint
[Skill] Created: - Websense
[Person] Created: Manuel Cuevas-Trisán (he, him, él)
[Skill] Created: Employment Law
[Skill] Created: Succession Planning
[Experience] Vice President for Human Resources at Harvard University · Full-time added.
[Organization] Harvard University · Full-time added.
[Experience] Vice President & Chief Human Resources Officer at Northwestern University · Full-time added.
[Organization] Northwestern University · Full-time added.
[Experience] CHRO and Corporate Vice President, Employment Law & Data Protection at Motorola Solutions added.
[Organization] Motorola Solutions added.
[Skill] Created: - Internal investigations
[Skill] Created: - GDPR-compliant data protection program
[Experience] Vice President, Litigation, Data Protection & Employment Law at Motorola Solutions added.
[Experience

[Person] Created: Forrest Gump
[Experience] Zero Trust Security Architect at Sophos added.
[Experience] IoT Security Specialist at Malwarebytes added.
[Experience] Intern at ESET added.
[Person] Created: Marty McFly
[Experience] Red Team Specialist at McAfee added.
[Experience] Forensic Investigator at Kaspersky Lab added.
[Experience] Intern at Zscaler added.
[Person] Created: Sarah Connor
[Skill] Created: Container Security
[Experience] Security Engineer at Cybereason added.
[Organization] Cybereason added.
[Experience] SOC Analyst at IBM Security added.
[Experience] Intern at Qualys added.
[Person] Created: Trinity
[Experience] Red Team Specialist at Zscaler added.
[Experience] Incident Responder at McAfee added.
[Experience] Intern at Trend Micro added.
[Person] Created: Morpheus
[Experience] DevSecOps Engineer at Bitdefender added.
[Experience] Cryptography Engineer at RSA Security added.
[Experience] Intern at Rapid7 added.
[Person] Created: Gandalf
[Experience] Blue Team Special

[Experience] Blue Team Specialist at Kaspersky Lab added.
[Experience] Intern at McAfee added.
[Person] Created: Kyle Broflovski
[Experience] Ethical Hacker at BlackBerry Cylance added.
[Experience] Security Engineer at Cybereason added.
[Experience] Intern at Palo Alto Networks added.
[Person] Created: Kenny McCormick
[Experience] Cyber Threat Intelligence Specialist at Sophos added.
[Experience] Security Researcher at FireEye added.
[Experience] Intern at Avast added.
[Person] Created: Patrick Bateman
[Experience] Ethical Hacker at Darktrace added.
[Experience] Blue Team Specialist at Malwarebytes added.
[Experience] Intern at Symantec added.
[Person] Created: Walter White
[Experience] Penetration Tester at BlackBerry Cylance added.
[Experience] Forensic Investigator at BlackBerry Cylance added.
[Experience] Intern at SentinelOne added.
[Person] Created: Jesse Pinkman
[Experience] Blue Team Specialist at RSA Security added.
[Experience] Cyber Threat Intelligence Specialist at Zscaler

In [14]:
from neo4j import GraphDatabase
import json
import os

# ---------------------
# CONFIGURATION
# ---------------------
uri = "bolt://localhost:7687"
username = "neo4j"
password = "root1234"
database = "neo4j"

# File path to your JSON file
file_path = "Cyber_security_experts_Manual_scraping_ENRICHED.json"

# ---------------------
# HELPER FUNCTION
# ---------------------
def sanitize_skill(raw_skill):
    """
    Remove leading dashes and spaces.
    Return None if the value is empty or "Null".
    """
    if not raw_skill or raw_skill.strip().lower() == "null":
        return None
    return raw_skill.lstrip("- ").strip()

# ---------------------
# TRANSACTION FUNCTION
# ---------------------
def add_expert_data(tx, file_path):
    with open(file_path, "r", encoding="utf-8") as jsonfile:
        data = json.load(jsonfile)
    
    # Process each expert record
    for expert in data:
        person_name = expert.get("name", "").strip()
        if not person_name:
            continue
        person_id = person_name.lower().replace(" ", "_")
        
        # Create/MERGE Person node
        query_person = """
            MERGE (p:Person {person_id: $person_id})
            ON CREATE SET p.name = $person_name
        """
        tx.run(query_person, person_id=person_id, person_name=person_name)
        print("Created/Merged Person: " + person_name)
        
        # Process top-level person skills (direct Person–HAS_SKILL relationship)
        for raw_skill in expert.get("skills", []):
            skill_name = sanitize_skill(raw_skill)
            if not skill_name:
                continue
            skill_id = skill_name.lower().replace(" ", "_")
            query_skill = """
                MERGE (s:Skill {skill_id: $skill_id})
                ON CREATE SET s.name = $skill_name
            """
            tx.run(query_skill, skill_id=skill_id, skill_name=skill_name)
            query_person_skill = """
                MATCH (p:Person {person_id: $person_id})
                MATCH (s:Skill {skill_id: $skill_id})
                MERGE (p)-[:HAS_SKILL]->(s)
            """
            tx.run(query_person_skill, person_id=person_id, skill_id=skill_id)
            print("Linked Person " + person_name + " with top-level Skill: " + skill_name)
        
        # Process experiences
        experiences = expert.get("experiences", [])
        for idx, exp in enumerate(experiences):
            exp_role = exp.get("role", "").strip()
            exp_workplace = exp.get("workplace", "").strip()
            exp_duration = exp.get("duration", "").strip()
            exp_description = exp.get("Description", "").strip()
            exp_id = f"{person_id}_exp_{idx}"
            
            query_exp = """
                MERGE (e:Experience {experience_id: $exp_id})
                ON CREATE SET e.role = $exp_role, e.duration = $exp_duration, e.description = $exp_description
            """
            tx.run(query_exp,
                   exp_id=exp_id,
                   exp_role=exp_role,
                   exp_duration=exp_duration,
                   exp_description=exp_description)
            query_person_exp = """
                MATCH (p:Person {person_id: $person_id})
                MATCH (e:Experience {experience_id: $exp_id})
                MERGE (p)-[:HAS_EXPERIENCE]->(e)
            """
            tx.run(query_person_exp, person_id=person_id, exp_id=exp_id)
            print("Linked Person " + person_name + " with Experience: " + exp_role)
            
            # Process Organization (if workplace available)
            if exp_workplace:
                org_id = exp_workplace.lower().replace(" ", "_")
                query_org = """
                    MERGE (o:Organization {organization_id: $org_id})
                    ON CREATE SET o.name = $org_name
                """
                tx.run(query_org, org_id=org_id, org_name=exp_workplace)
                query_exp_org = """
                    MATCH (e:Experience {experience_id: $exp_id})
                    MATCH (o:Organization {organization_id: $org_id})
                    MERGE (e)-[:AT_ORGANIZATION]->(o)
                """
                tx.run(query_exp_org, exp_id=exp_id, org_id=org_id)
                print("Linked Experience " + exp_role + " with Organization: " + exp_workplace)
            
            # Process experience-level skills (Experience–USED_SKILL)
            for raw_skill in exp.get("skills_extracted", []):
                skill_name = sanitize_skill(raw_skill)
                if not skill_name:
                    continue
                skill_id = skill_name.lower().replace(" ", "_")
                tx.run(query_skill, skill_id=skill_id, skill_name=skill_name)
                query_exp_skill = """
                    MATCH (e:Experience {experience_id: $exp_id})
                    MATCH (s:Skill {skill_id: $skill_id})
                    MERGE (e)-[:USED_SKILL]->(s)
                """
                tx.run(query_exp_skill, exp_id=exp_id, skill_id=skill_id)
                print("Linked Experience " + exp_role + " with Skill: " + skill_name)

# ---------------------
# MAIN EXECUTION
# ---------------------
try:
    driver = GraphDatabase.driver(uri, auth=(username, password), database=database)
    with driver.session() as session:
        session.write_transaction(add_expert_data, file_path)
    driver.close()
except Exception as e:
    print(f"Error: {e}")


/var/folders/zq/w3gtlndx0qsdt92gmtmqdnkm0000gn/T/ipykernel_91285/2973888221.py:132: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(add_expert_data, file_path)


Created/Merged Person: Adam Evans
Linked Person Adam Evans with top-level Skill: Web Development
Linked Person Adam Evans with top-level Skill: Ethical Hacking
Linked Person Adam Evans with top-level Skill: Penetration Testing
Linked Person Adam Evans with top-level Skill: Malware Analysis
Linked Person Adam Evans with top-level Skill: Reverse Engineering
Linked Person Adam Evans with top-level Skill: Vulnerability Assessment
Linked Person Adam Evans with top-level Skill: Network Security
Linked Person Adam Evans with top-level Skill: Firewall Management
Linked Person Adam Evans with top-level Skill: Intrusion Detection
Linked Person Adam Evans with top-level Skill: System Administration
Linked Person Adam Evans with top-level Skill: Network Administration
Linked Person Adam Evans with top-level Skill: Security Operations
Linked Person Adam Evans with top-level Skill: Storage Area Networks
Linked Person Adam Evans with top-level Skill: Backtrack
Linked Person Adam Evans with top-level 

Linked Person Alan Mitchell with top-level Skill: Fedramp
Linked Person Alan Mitchell with top-level Skill: HIPPA
Linked Person Alan Mitchell with top-level Skill: Government Liaison
Linked Person Alan Mitchell with top-level Skill: Law Enforcement
Linked Person Alan Mitchell with top-level Skill: Privacy Law
Linked Person Alan Mitchell with top-level Skill: Manufacturing Engineering
Linked Person Alan Mitchell with top-level Skill: U.S. Health Insurance Portability and Accountability Act (HIPAA)
Linked Person Alan Mitchell with top-level Skill: Disaster Recovery
Linked Person Alan Mitchell with top-level Skill: Mergers and Acquisitions
Linked Person Alan Mitchell with Experience: Vice President and Global Chief Information Security Officer/Corporate Security Officer
Linked Experience Vice President and Global Chief Information Security Officer/Corporate Security Officer with Organization: Celanese
Linked Person Alan Mitchell with Experience: Vice President and Global Chief Security Of

Linked Person Andy Matthiesen with top-level Skill: Information Technology
Linked Person Andy Matthiesen with top-level Skill: Software Development Life Cycle (SDLC)
Linked Person Andy Matthiesen with Experience: Chief Security Officer
Linked Experience Chief Security Officer with Organization: Gen Re
Linked Person Andy Matthiesen with Experience: CTO
Linked Experience CTO with Organization: FT Search
Linked Person Andy Matthiesen with Experience: Executive Director
Linked Experience Executive Director with Organization: American Red Cross
Linked Person Andy Matthiesen with Experience: VP Product Development
Linked Experience VP Product Development with Organization: DecisionView
Linked Person Andy Matthiesen with Experience: Senior Director of Technology
Linked Experience Senior Director of Technology with Organization: Rx Remedy
Linked Person Andy Matthiesen with Experience: Assistant Director, Information Technology
Linked Experience Assistant Director, Information Technology with O

Created/Merged Person: Aaron J. Goodwin
Linked Person Aaron J. Goodwin with top-level Skill: Cyber-security
Linked Person Aaron J. Goodwin with top-level Skill: Certified Information Systems Security Professional (CISSP)
Linked Person Aaron J. Goodwin with top-level Skill: Certified Chief Information Security Officer (CCISO)
Linked Person Aaron J. Goodwin with top-level Skill: Build Strong Relationships
Linked Person Aaron J. Goodwin with top-level Skill: Security Operations Center
Linked Person Aaron J. Goodwin with top-level Skill: Executive Leadership
Linked Person Aaron J. Goodwin with top-level Skill: Information Security Management
Linked Person Aaron J. Goodwin with top-level Skill: Vulnerability Assessment
Linked Person Aaron J. Goodwin with top-level Skill: Information Technology
Linked Person Aaron J. Goodwin with top-level Skill: Vulnerability
Linked Person Aaron J. Goodwin with top-level Skill: Network Security
Linked Person Aaron J. Goodwin with top-level Skill: Informatio

...

Linked Experience Sr. Architect with Organization: ZeniMax Online Studios
Linked Person Rob Gil with Experience: Infrastructure Architect
Linked Experience Infrastructure Architect with Organization: OTC Markets Group
Linked Person Rob Gil with Experience: Systems Engineer
Linked Experience Systems Engineer with Organization: Liquidnet
Linked Person Rob Gil with Experience: Linux Systems Administrator
Linked Experience Linux Systems Administrator with Organization: American Home Mortgage
Linked Person Rob Gil with Experience: Owner
Linked Experience Owner with Organization: REM5
Linked Person Rob Gil with Experience: Systems Administrator
Linked Experience Systems Administrator with Organization: Desktop Solutions Software, Inc.
Linked Person Rob Gil with Experience: Systems Administrator
Linked Experience Systems Administrator with Organization: Body Building Discount Inc.
Created/Merged Person: HD Moore
Linked Person HD Moore with top-level Skill: Penetration Testing
Linked Person HD

Linked Person Charlie Miller with Experience: Principal Analyst, Software Security
Linked Experience Principal Analyst, Software Security with Organization: Independent Security Evaluators
Linked Person Charlie Miller with Experience: Senior Security Architect
Linked Experience Senior Security Architect with Organization: Financial Networks Incorporated
Linked Person Charlie Miller with Experience: Global Network Exploitation Analyst
Linked Experience Global Network Exploitation Analyst with Organization: National Security Agency
Created/Merged Person: Aeva Black
Linked Person Aeva Black with top-level Skill: Open Source
Linked Person Aeva Black with top-level Skill: Cloud Computing
Linked Person Aeva Black with top-level Skill: System Architecture
Linked Person Aeva Black with top-level Skill: High Availability
Linked Person Aeva Black with top-level Skill: Virtualization
Linked Person Aeva Black with top-level Skill: Scalability
Linked Person Aeva Black with top-level Skill: Linux Sy

Linked Person Christopher Butera with top-level Skill: Linux System Administration
Linked Person Christopher Butera with top-level Skill: Network Administration
Linked Person Christopher Butera with top-level Skill: Database Administration
Linked Person Christopher Butera with top-level Skill: CISSP
Linked Person Christopher Butera with top-level Skill: OS X
Linked Person Christopher Butera with top-level Skill: Oracle
Linked Person Christopher Butera with top-level Skill: Network Security
Linked Person Christopher Butera with top-level Skill: Apache
Linked Person Christopher Butera with top-level Skill: Cloud Computing
Linked Person Christopher Butera with Experience: Senior Technical Director for Cyber
Linked Experience Senior Technical Director for Cyber with Organization: Cybersecurity and Infrastructure Security Agency
Linked Person Christopher Butera with Experience: Acting Deputy Executive Assistant Director for Cyber
Linked Experience Acting Deputy Executive Assistant Director 

Linked Person Jack Cable with top-level Skill: jQuery
Linked Person Jack Cable with top-level Skill: Node.js
Linked Person Jack Cable with top-level Skill: PHP
Linked Person Jack Cable with top-level Skill: C++
Linked Person Jack Cable with top-level Skill: x86 Assembly
Linked Person Jack Cable with top-level Skill: API Development
Linked Person Jack Cable with top-level Skill: Ethical Hacking
Linked Person Jack Cable with top-level Skill: Network Security
Linked Person Jack Cable with top-level Skill: Information Security
Linked Person Jack Cable with top-level Skill: Web Development
Linked Person Jack Cable with top-level Skill: MySQL
Linked Person Jack Cable with top-level Skill: Programming
Linked Person Jack Cable with top-level Skill: Databases
Linked Person Jack Cable with top-level Skill: iOS Development
Linked Person Jack Cable with top-level Skill: Algorithms
Linked Person Jack Cable with top-level Skill: Java
Linked Person Jack Cable with top-level Skill: Security
Linked Per

Linked Experience Co-host with Organization: Security Hype
Linked Person Bob Lord with Experience: Senior Director, Directory and Security Engineering
Linked Experience Senior Director, Directory and Security Engineering with Organization: Red Hat
Linked Person Bob Lord with Experience: Senior Director, Client and Security Engineering
Linked Experience Senior Director, Client and Security Engineering with Organization: AOL/Time Warner
Linked Person Bob Lord with Experience: Director, Security Engineering
Linked Experience Director, Security Engineering with Organization: AOL/Time Warner
Linked Person Bob Lord with Experience: Manager, Corporate Electronic Security
Linked Experience Manager, Corporate Electronic Security with Organization: Netscape Communications Corp.
Linked Person Bob Lord with Experience: President and Co-founder
Linked Experience President and Co-founder with Organization: Emerge Consulting
Linked Person Bob Lord with Experience: IS Manager
Linked Experience IS Mana

In [15]:
print("done")

done


In [14]:
from neo4j import GraphDatabase
import json
import os

# ---------------------
# CONFIGURATION
# ---------------------
uri = "bolt://localhost:7687"
username = "neo4j"
password = "root1234"
database = "neo4j"

# File path to your JSON file
file_path = "CyberSecuirtyResearchers_2024_ENRICHED.json"

# ---------------------
# HELPER FUNCTION
# ---------------------
def sanitize_skill(raw_skill):
    """
    Remove leading dashes and spaces.
    Return None if the value is empty or "Null".
    """
    if not raw_skill or raw_skill.strip().lower() == "null":
        return None
    return raw_skill.lstrip("- ").strip()

# ---------------------
# TRANSACTION FUNCTION
# ---------------------
def add_expert_data(tx, file_path):
    
    with open(file_path, "r", encoding="utf-8") as jsonfile:
        data = json.load(jsonfile)
    
    query_skill = """
        MERGE (s:Skill {skill_id: $skill_id})
        ON CREATE SET s.name = $skill_name
    """
    # Process each expert record
    for expert in data:
        person_name = expert.get("name", "").strip()
        if not person_name:
            continue
        person_id = person_name.lower().replace(" ", "_")
        
        # Create/MERGE Person node
        query_person = """
            MERGE (p:Person {person_id: $person_id})
            ON CREATE SET p.name = $person_name
        """
        tx.run(query_person, person_id=person_id, person_name=person_name)
        print("Created/Merged Person: " + person_name)
        
        # Process top-level person skills (direct Person–HAS_SKILL relationship)
        for raw_skill in expert.get("skills", []):
            skill_name = sanitize_skill(raw_skill)
            if not skill_name:
                continue
            skill_id = skill_name.lower().replace(" ", "_")
            query_skill = """
                MERGE (s:Skill {skill_id: $skill_id})
                ON CREATE SET s.name = $skill_name
            """
            tx.run(query_skill, skill_id=skill_id, skill_name=skill_name)
            query_person_skill = """
                MATCH (p:Person {person_id: $person_id})
                MATCH (s:Skill {skill_id: $skill_id})
                MERGE (p)-[:HAS_SKILL]->(s)
            """
            tx.run(query_person_skill, person_id=person_id, skill_id=skill_id)
            print("Linked Person " + person_name + " with top-level Skill: " + skill_name)
        
        # Process experiences
        experiences = expert.get("experiences", [])
        for idx, exp in enumerate(experiences):
            exp_role = exp.get("role", "").strip()
            exp_workplace = exp.get("workplace", "").strip()
            exp_duration = str(exp.get("duration", "")).strip()
            exp_description = exp.get("Description", "").strip()
            exp_id = f"{person_id}_exp_{idx}"
            
            query_exp = """
                MERGE (e:Experience {experience_id: $exp_id})
                ON CREATE SET e.role = $exp_role, e.duration = $exp_duration, e.description = $exp_description
            """
            tx.run(query_exp,
                   exp_id=exp_id,
                   exp_role=exp_role,
                   exp_duration=exp_duration,
                   exp_description=exp_description)
            query_person_exp = """
                MATCH (p:Person {person_id: $person_id})
                MATCH (e:Experience {experience_id: $exp_id})
                MERGE (p)-[:HAS_EXPERIENCE]->(e)
            """
            tx.run(query_person_exp, person_id=person_id, exp_id=exp_id)
            print("Linked Person " + person_name + " with Experience: " + exp_role)
            
            # Process Organization (if workplace available)
            if exp_workplace:
                org_id = exp_workplace.lower().replace(" ", "_")
                query_org = """
                    MERGE (o:Organization {organization_id: $org_id})
                    ON CREATE SET o.name = $org_name
                """
                tx.run(query_org, org_id=org_id, org_name=exp_workplace)
                query_exp_org = """
                    MATCH (e:Experience {experience_id: $exp_id})
                    MATCH (o:Organization {organization_id: $org_id})
                    MERGE (e)-[:AT_ORGANIZATION]->(o)
                """
                tx.run(query_exp_org, exp_id=exp_id, org_id=org_id)
                print("Linked Experience " + exp_role + " with Organization: " + exp_workplace)
            
            # Process experience-level skills (Experience–USED_SKILL)
            for raw_skill in exp.get("skills_extracted", []):
                skill_name = sanitize_skill(raw_skill)
                if not skill_name:
                    continue
                skill_id = skill_name.lower().replace(" ", "_")
                tx.run(query_skill, skill_id=skill_id, skill_name=skill_name)
                query_exp_skill = """
                    MATCH (e:Experience {experience_id: $exp_id})
                    MATCH (s:Skill {skill_id: $skill_id})
                    MERGE (e)-[:USED_SKILL]->(s)
                """
                tx.run(query_exp_skill, exp_id=exp_id, skill_id=skill_id)
                print("Linked Experience " + exp_role + " with Skill: " + skill_name)

# ---------------------
# MAIN EXECUTION
# ---------------------
try:
    driver = GraphDatabase.driver(uri, auth=(username, password), database=database)
    with driver.session() as session:
        session.write_transaction(add_expert_data, file_path)
    driver.close()
except Exception as e:
    print(f"Error: {e}")


/var/folders/zq/w3gtlndx0qsdt92gmtmqdnkm0000gn/T/ipykernel_92052/873267737.py:137: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(add_expert_data, file_path)


Created/Merged Person: Jorge   Crichigno
Linked Person Jorge   Crichigno with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: University of South Carolina at Columbia
Linked Experience Principal Investigator with Skill: Network Security
Linked Experience Principal Investigator with Skill: ML (Machine Learning) for Malware Detection and Classification
Linked Experience Principal Investigator with Skill: P4 Programmable Data Planes
Linked Experience Principal Investigator with Skill: SmartNICs
Linked Experience Principal Investigator with Skill: Deep Packet Inspection (DPI)
Linked Experience Principal Investigator with Skill: Domain Name System (DNS) Packet Analysis
Linked Experience Principal Investigator with Skill: Real-time Traffic Monitoring
Linked Experience Principal Investigator with Skill: Malware Detection and Classification
Linked Experience Principal Investigator with Skill: Feature Extraction for Encrypted DNS Packets
Linked Exp

Linked Experience Co-Principal Investigator with Skill: Interoperable secure operations for wireless devices
Created/Merged Person: Rittika   Shamsuddin
Linked Person Rittika   Shamsuddin with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: Oklahoma State University
Linked Experience Principal Investigator with Skill: Security concepts
Linked Experience Principal Investigator with Skill: Data privacy
Linked Experience Principal Investigator with Skill: System vulnerabilities
Linked Experience Principal Investigator with Skill: Ethical AI use
Linked Experience Principal Investigator with Skill: Communications networks security
Linked Experience Principal Investigator with Skill: Network analysis
Linked Experience Principal Investigator with Skill: Network simulators
Linked Experience Principal Investigator with Skill: Vehicular network security
Linked Experience Principal Investigator with Skill: Cyber threat intelligence
Created/Merged Per

Linked Experience Principal Investigator with Skill: Cryptographic primitives
Linked Experience Principal Investigator with Skill: Side-channel vulnerability analysis
Linked Experience Principal Investigator with Skill: Countermeasures using randomization techniques in hardware
Linked Experience Principal Investigator with Skill: Hardware security
Created/Merged Person: Taeho   Jung
Linked Person Taeho   Jung with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: University of Notre Dame
Linked Experience Principal Investigator with Skill: Secure computation
Linked Experience Principal Investigator with Skill: Homomorphic computations
Linked Experience Principal Investigator with Skill: Confidential computing
Linked Experience Principal Investigator with Skill: Trusted execution environments (TEE)
Linked Experience Principal Investigator with Skill: Zero-knowledge enclave verification
Linked Experience Principal Investigator with Skill: End-

Linked Experience Principal Investigator with Skill: Systematic Fuzzing
Linked Experience Principal Investigator with Skill: Cross-language Support
Linked Experience Principal Investigator with Skill: Automated Synthesis of Fuzzing Harnesses
Linked Experience Principal Investigator with Skill: Automated Mining of Formal Input Specifications
Created/Merged Person: Jack W Davidson
Linked Person Jack W Davidson with Experience: Co-Principal Investigator
Linked Experience Co-Principal Investigator with Organization: University of Utah
Linked Experience Co-Principal Investigator with Skill: Fuzz Testing
Linked Experience Co-Principal Investigator with Skill: Vulnerability-finding
Linked Experience Co-Principal Investigator with Skill: Systematic Fuzzing
Linked Experience Co-Principal Investigator with Skill: Cross-language Support
Linked Experience Co-Principal Investigator with Skill: Automated Synthesis of Fuzzing Harnesses
Linked Experience Co-Principal Investigator with Skill: Automated

...

Linked Person Suxia   Cui with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: Prairie View A & M University
Linked Experience Principal Investigator with Skill: Security-aware resource management
Linked Experience Principal Investigator with Skill: Attack-facing approaches
Linked Experience Principal Investigator with Skill: Reinforcement-based formulation for optimization
Linked Experience Principal Investigator with Skill: Data-driven solutions for critical attack factors
Linked Experience Principal Investigator with Skill: Cybersecurity research enhancement
Linked Experience Principal Investigator with Skill: Security-centric resource management
Created/Merged Person: Chia-Che   Tsai
Linked Person Chia-Che   Tsai with Experience: Co-Principal Investigator
Linked Experience Co-Principal Investigator with Organization: Prairie View A & M University
Linked Experience Co-Principal Investigator with Skill: Security-aware resource management

Linked Experience Co-Principal Investigator with Skill: Cyber Threat Simulation
Linked Experience Co-Principal Investigator with Skill: Penetration Skills
Linked Experience Co-Principal Investigator with Skill: Cybersecurity Auditing Skills
Created/Merged Person: Jeremy E Guinn
Linked Person Jeremy E Guinn with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: United Tribes Technical College
Linked Experience Principal Investigator with Skill: Cybersecurity
Created/Merged Person: Niraj K Jha
Linked Person Niraj K Jha with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: Princeton University
Linked Experience Principal Investigator with Skill: Healthcare information security
Linked Experience Principal Investigator with Skill: Data security
Linked Experience Principal Investigator with Skill: Cybersecurity risk analysis
Linked Experience Principal Investigator with Skill: Vulnerability analysis
Li

Linked Experience Co-Principal Investigator with Skill: Network security protocols and standards
Linked Experience Co-Principal Investigator with Skill: Public-key infrastructure (PKI)
Linked Experience Co-Principal Investigator with Skill: Symmetric-key cryptography
Linked Experience Co-Principal Investigator with Skill: Access control
Linked Experience Co-Principal Investigator with Skill: Key management
Linked Experience Co-Principal Investigator with Skill: Post-quantum cryptography (PQC)
Linked Experience Co-Principal Investigator with Skill: Symmetric encryption algorithms
Linked Experience Co-Principal Investigator with Skill: Distributed trust
Linked Experience Co-Principal Investigator with Skill: Secure multi-party computation
Linked Experience Co-Principal Investigator with Skill: Decentralized architectures
Linked Experience Co-Principal Investigator with Skill: Compromise-resilient PKIs
Linked Experience Co-Principal Investigator with Skill: Certificateless credentials
Lin

Linked Experience Principal Investigator with Skill: Security vulnerabilities analysis
Linked Experience Principal Investigator with Skill: Protection mechanisms development
Linked Experience Principal Investigator with Skill: Side-channel leakage mechanism investigation
Linked Experience Principal Investigator with Skill: Memory-access contention analysis
Linked Experience Principal Investigator with Skill: Attacker framework development
Linked Experience Principal Investigator with Skill: Memory-contention-based signature extraction
Linked Experience Principal Investigator with Skill: Profiling and analysis techniques for machine learning and AI workloads
Linked Experience Principal Investigator with Skill: Reverse-engineering attack creation
Linked Experience Principal Investigator with Skill: Information extraction attack creation
Linked Experience Principal Investigator with Skill: Denial-of-service attack creation
Linked Experience Principal Investigator with Skill: Mitigation de

Linked Experience Principal Investigator with Skill: Verified bi-directional translation
Linked Experience Principal Investigator with Skill: Security threats identification and mitigation
Linked Experience Principal Investigator with Skill: Embedded systems security
Linked Experience Principal Investigator with Skill: Industrial control systems safety and security
Created/Merged Person: Phuong M Cao
Linked Person Phuong M Cao with Experience: Principal Investigator
Linked Experience Principal Investigator with Organization: University of Illinois at Urbana-Champaign
Linked Experience Principal Investigator with Skill: Quantum-resistant cryptography
Linked Experience Principal Investigator with Skill: Post-quantum cryptography (PQC)
Linked Experience Principal Investigator with Skill: Network architecture
Linked Experience Principal Investigator with Skill: Encryption adoption tracking
Linked Experience Principal Investigator with Skill: Migration to new encryption
Created/Merged Perso